
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# 임베딩과 AI Search

## 서론

어떤 회수 증강 생성(RAG) 시스템의 효과는 한 가지 중요한 요소에 달려 있습니다: 회수 파이프라인의 품질. 관련 정보를 가져오기 전에, 먼저 비구조화 텍스트를 **임베딩**라고 불리는 수치 표현으로 변환하여 특수 **벡터 데이터베이스**에 저장해야 합니다. 이번 수업에서는 데이터 준비 전체 수명 주기를 탐구합니다—**임베딩 모델**이 텍스트를 벡터로 변환하는 방법부터 효율적인 검색을 위한 **벡터 유사도 알고리즘**을 활용하는 것까지. 또한 하이브리드 검색과 재랭킹 같은 첨단 검색 기법도 검토하여 결과 품질을 크게 향상시킬 것입니다. 마지막으로, **Databricks AI Search**가 Databricks Data Intelligence Platform 내에서 이러한 구성 요소를 어떻게 통합하여 안전한 서버리스 벡터 데이터베이스 솔루션을 제공하는지 알아볼 것입니다.

## 수업 목표

이 수업이 끝날 때쯤이면 다음과 같은 행동을 할 수 있게 될 것입니다:

* 임베딩 벡터의 키 특성을 식별하고 적절한 임베딩 모델 선택 기준을 평가합니다.
* 유사성 검색, 전체 텍스트 검색, 하이브리드 검색 방법론을 비교하여 최적의 검색 전략을 결정합니다.
* 생산 환경에서 정확 탐색 알고리즘과 근사 탐색 알고리즘 간의 상충 관계를 분석합니다.
* 재랭킹이 어떻게 문맥 정밀도를 향상시키고 RAG 적용 시 환각을 줄인다고 설명하세요.
* Databricks AI Search의 인제스션 모드, 거버넌스 모델, 아키텍처적 이점을 설명하세요.

## A. 임베딩의 핵심 개념

이 섹션에서는 임베딩, 즉 현대 정보 검색의 수학적 중추를 다루는 데 필요한 기초 지식을 확립할 것입니다. 우리는 **비구조화된 텍스트가 벡터 표현으로 변환되는 방법**을 탐구하고, 특정 도메인에 적합한 모델 선택이 왜 중요한지 살펴보며, 쿼리와 문서 공간 간의 일치가 얼마나 중요한지 이해할 것입니다.

### A1. 임베딩 정의

임베딩은 일반적으로 딥러닝 모델에 의해 생성되는 내용의 수치적 표현입니다. 이 모델들은 고차원 비구조화 데이터(예: 텍스트)를 저차원 벡터, 즉 의미적 의미를 포착하는 부동소수점 배열로 변환합니다. 임베딩을 강력하게 만드는 키 특성은 유사한 개념들을 벡터 공간에서 가깝게 매핑할 수 있다는 점입니다. 의미와 관련된 단어나 구들이 서로 가까이 클러스터되어 있어, 정확한 키워드가 일치하지 않아도 개념적 관계를 식별할 수 있게 합니다.

### A2. 멀티모달 맥락

**이 강의는 비구조화 텍스트에 초점**을 맞추고 있지만, 임베딩이 단어 그 이상으로 확장된다는 점을 참고할 가치가 있습니다. GPT-4o와 Gemini 1.5와 같은 멀티모달 모델은 이미지, 오디오, 텍스트를 통합된 벡터 공간에 처리하고 삽입할 수 있습니다. 이 기능은 교차 모달 검색 시나리오를 열어줍니다—텍스트 쿼리를 사용해 의미적으로 관련 있는 이미지를 찾거나, 설명이 포함된 오디오 콘텐츠를 검색하는 것과 같습니다.

### A3. 임베딩 모델

임베딩 모델은 텍스트, 이미지, 오디오와 같은 고차원 비정형 데이터를 저차원 수치 벡터로 변환하도록 설계된 특수한 기계 학습 모델(일반적으로 심층 신경망)입니다. **이를 인간이 읽을 수 있는 콘텐츠를 기계가 읽을 수 있는 부동 소수점 숫자 목록으로 변환하는 번역기로 생각하면 됩니다. 이 모델은 유사한 의미를 가진 입력값이 수학적으로 서로 가까운 벡터를 생성하도록 보장합니다.**

적절한 임베딩 모델 선택은 검색 품질에 영향을 미치는 중요한 아키텍처적 결정입니다. 다음 키 요소들을 고려하십시오:

- **어휘 크기 및 도메인:** 일부 모델은 일반 웹 텍스트를 기반으로 훈련하는 반면, 다른 모델은 금융, 의학, 법률 문서 등 특정 분야에 특화되어 있습니다. 도메인 특정 모델은 종종 특화된 콘텐츠에 대해 우수한 결과를 제공합니다.
- **컨텍스트 Window:** 모든 모델은 최대 입력 토큰 한도를 가지고 있습니다. 이 제한을 초과하는 텍스트는 잘리거나 무시되어, 긴 문서에서는 효과적인 청크 전략이 필수적입니다.
- **치수:** 더 높은 차원의 벡터(더 큰 배열)는 더 많은 뉘앙스와 의미론적 세부사항을 포착하지만, 저장 비용과 검색 지연 시간을 증가시킵니다. 정밀도 요구와 운영 제약 조건의 균형을 맞추세요.

<!-- <img src="../Includes/images/03-vectorization.png" alt="Vectorization process illustrated" /> -->
![03-vectorization](https://files.training.databricks.com/binder/prod_main/building-retrieval-agents-on-databricks-ko_kr-1.0.1/images/03-vectorization.png)

*그림 1. 이 그림은 임베딩 모델이 데이터 청크를 처리하여 벡터를 생성하는 방식을 보여줍니다. 입력 데이터가 모델의 컨텍스트 Window 한도를 초과하면 초과 내용이 생략되어 결과 임베딩의 완성도에 영향을 줄 수 있습니다.*

### A4. 임베딩 정렬

검색이 효과적으로 작동하려면, 임베딩 모델이 동일한 벡터 공간 내에 소스 문서와 사용자 쿼리를 모두 표현해야 합니다. 모델이 주로 장기 문서로 학습되지만, 애플리케이션이 짧고 비공식적인 쿼리를 사용한다면, 벡터 표현이 잘 맞지 않아 검색 결과가 좋지 않을 수 있습니다. 최선의 방법은 간단합니다: **문서 인덱싱과 쿼리 처리 모두에 동일한 임베딩 모델을 사용합니다**. 이로 인해 임베딩이 동일한 수학적 공간에 존재하며 의미 있게 비교할 수 있습니다.

## B. 벡터 저장 및 탐색 메커니즘

비정형 데이터를 임베딩으로 변환한 후에는 고차원 벡터를 처리하고 효율적인 유사도 쿼리를 수행할 수 있는 특수 저장소가 필요합니다. 이 섹션에서는 벡터 데이터베이스의 독특한 아키텍처와 전통적인 관계형 시스템과의 차이점을 살펴보겠습니다. 또한 의미적으로 관련 있는 정보를 대규모로 검색하기 위해 사용되는 검색 알고리즘과 지표도 탐구할 것입니다.

### B1. 벡터 데이터베이스의 역할

벡터 데이터베이스는 고차원 벡터를 효율적으로 저장하고 검색하기 위해 특별히 구축되었습니다. 전통적인 데이터베이스가 정확한 일치를 위해 설계된 것(예를 들어, SQL WHERE 절)과 달리, 벡터 데이터베이스는 동일하지 않지만 개념적으로 관련이 있는 항목을 찾는 유사성 검색에서 뛰어납니다. CRUD(생성-읽기-업데이트-삭제) 작업과 같은 표준 데이터베이스 기능을 유지하면서 벡터 연산에 최적화된 특수화된 인덱싱 구조를 도입합니다.

### B2. 검색 방법

서로 다른 검색 방법은 서로 다른 검색 요구를 충족합니다:

- **유사도 검색:** 이 방법은 정확한 단어 매칭이 아닌 의미 상관관계를 기반으로 콘텐츠를 검색합니다. 이는 "불안 대처법"과 같은 자연어 쿼리가 "PTSD 대처"나 "스트레스 관리"와 같이 다른 용어를 사용할 수 있는 관련 결과를 생성할 수 있게 합니다.
- **전체 텍스트 검색:** 이 전통적인 방법은 키워드 매칭에 의존합니다. 이는 부품 번호, 제품 코드, 고유명사 같은 특정 용어를 찾는 데는 뛰어나지만, 의미적 의도나 동의어 인식에는 실패합니다.
- **하이브리드 검색:** 이 강력한 접근법은 벡터 유사도 검색과 키워드 기반 검색을 결합합니다. 의미론적 이해와 정확한 키워드 매칭을 모두 활용함으로써, 하이브리드 검색은 일반적으로 두 방법 중 어느 쪽보다 더 높은 검색 정확도를 제공합니다.

### B3. 거리 및 유사도 지표

두 벡터가 얼마나 "유사한지"를 판단하기 위해, 우리는 서로 다른 검색 시나리오에 적합한 두 가지 주요 메트릭인 **거리 메트릭**와 **유사성 메트릭**를 사용합니다. **거리 메트릭**는 두 벡터가 공간에서 얼마나 떨어져 있는지 정량화하고 싶을 때 사용됩니다—클러스터링, 이상치 검출, 또는 크기가 중요한 경우에 이상적입니다. **유사도 메트릭**은 두 벡터가 방향적으로 얼마나 밀접하게 정렬되어 있는지 알고 싶을 때 사용되며, 의미론적 검색, 문서 검색, 그리고 의미가 규모보다 더 중요한 대부분의 NLP 응용 분야에 이상적입니다.

**거리 메트릭**  
- **유클리드 거리 (L2):** 벡터 공간 내 두 점 사이의 직선 거리를 측정합니다. *낮은* 값은 벡터가 더 비슷하다는 뜻입니다. 클러스터링이나 이상 탐지 등 모든 차원의 절대 차이에 관심이 있을 때 이 기능을 사용하세요.
- **맨해튼 거리 (L1):** 모든 차원에 걸친 절대 차이를 합산합니다. *낮은* 값은 벡터가 더 가까워진다는 뜻입니다. 이는 격자 기반이나 희소 데이터처럼 각 축을 따라 차이가 똑같이 중요할 때 유용합니다.

**유사도 지표**  
- **코사인 유사성:** 두 벡터 사이의 각도의 코사인을 측정합니다. *높은* 점수는 더 큰 유사성을 의미합니다. 이 지표는 텍스트 임베딩에서 가장 인기 있는 지표인데, 크기보다는 방향(의미적 의미)에 초점을 맞추어 문서의 길이나 규모 차이에 강합니다.


<!-- <img src="../Includes/images/03-vector-similarities.png" alt="Distance and similarity metrics" /> -->

![03-vector-similarities](https://files.training.databricks.com/binder/prod_main/building-retrieval-agents-on-databricks-ko_kr-1.0.1/images/03-vector-similarities.png)

*그림 2. 이 다이어그램은 위에 나열된 거리 지표와 유사성 지표를 시각화합니다*

### B4. 검색 전략

정확도와 성능을 균형 있게 조절하는 두 가지 주요 전략:

- **K-Nearest Neighbors (KNN):** 데이터베이스 내 쿼리 벡터와 *모든* 벡터 사이의 거리를 계산하는 정확한 검색 방법. 매우 정확하지만 이는 계산 비용이 많이 들고 대규모 데이터셋에 잘 대응하지 못합니다—수백만 개의 문서와 쿼리를 하나씩 비교해 보세요.
- **Approximate Nearest Neighbors (ANN):** 약간의 정확도를 희생하고 극적인 속도 향상을 얻는 전략. ANN는 **HNSW**(계층적 탐색 가능한 작은 세계)나 **FAISS**(페이스북 AI 유사도 검색)와 같은 정교한 인덱싱 알고리즘을 사용하여 벡터 공간을 효율적으로 탐색하며, 벡터 일부만 검사하면서도 매우 관련성 높은 결과를 찾습니다.

## C. 정밀도, 품질, 그리고 재순위

벡터 데이터베이스는 의미적으로 유사한 내용을 찾는 강력한 메커니즘을 제공하지만, 한계가 없는 것은 아닙니다. 이 섹션에서는 임베딩 품질의 미묘한 차이와 수학적 유사성과 진정한 의미적 관련성 사이의 잠재적 간극을 다룰 것입니다. 또한 검색 후 중요한 단계로서 재랭킹을 도입하여 결과를 정제하고 언어 모델에 제공되는 맥락의 정확성을 향상시킬 것입니다.

### C1. 임베딩 품질과 한계

중요한 인사이트: **유사성이 의미적 관련성을 의미하는 것은 아닙니다**. 어떤 문서는 벡터 공간에서 수학적으로 쿼리와 비슷하지만, 사실상 무관하거나 맥락상 부적절할 수 있습니다. 임베딩 품질은 모델, 그 학습 데이터, 그리고 그것이 특정 도메인과 얼마나 잘 맞는지에 크게 좌우됩니다. 부적절하게 준비된 데이터나 모델의 학습 말뭉치와 애플리케이션 콘텐츠 간의 불일치는 검색 성능 저하와 '분실' 정보로 이어질 수 있습니다.

또 다른 일반적인 시나리오는 유사도 검색의 모든 결과를 사용하는 대신 일부 문서를 선택하는 것입니다. 문서 수를 제한해야 할 때—토큰 제약이나 처리 비용 때문일 수 있다—가장 관련성 높은 문서가 상단에 오르도록 해야 합니다. 이럴 때 순위 조정이 필수적입니다.

### C2. 순위 조정 과정

초기 검색의 정밀도 격차를 메우기 위해 파이프라인에 **reranker**를 추가합니다:

1. **초기 검색:** 벡터 저장소는 빠른 ANN 알고리즘을 사용하여 일반적으로 상위 20개에서 50개 사이의 후보 문서를 폭넓게 검색합니다.
1. **재순위 조정:** 특수 모델(종종 크로스 인코더)은 각 후보 문서의 실제 관련성을 특정 쿼리와 비교하여 그 관계를 상세히 고려하여 평가합니다.
1. **재정렬:** 문서는 재랭커의 관련성 점수에 따라 재정렬되며, 언어 모델이 처리할 가장 중요한 정보가 상단에 배치됩니다.

<!-- <img src="../Includes/images/03-reranking.png" alt="Reranking process" width="500" /> -->

![03-reranking](https://files.training.databricks.com/binder/prod_main/building-retrieval-agents-on-databricks-ko_kr-1.0.1/images/03-reranking.png)

*그림 3. 이 도표는 리랭킹 과정이 어떻게 작동하는지 보여줍니다*

### C3. 장점과 트레이드오프

리랭킹은 중요한 고려사항을 도입합니다:

- **이점:** 리랭킹은 언어 모델에 제공되는 맥락의 정확성을 크게 향상시켜 환각을 직접 줄이고 응답 질을 향상시킵니다. 초기 검색 결과를 정제함으로써 가장 적합한 정보가 생성 단계에 도달하도록 보장합니다.
- **트레이드오프:** 리랭커를 추가하면 검색 파이프라인에서 지연과 비용이 모두 증가합니다. 재순위 모델은 쿼리 및 후보 문서를 실시간으로 처리해야 하므로 계산 오버헤드가 증가합니다. 이 비용을 특정 사용 사례에 맞는 품질 향상과 균형을 맞추세요.

## D. Databricks AI Search - 특징과 아키텍처

견고한 벡터 데이터베이스 인프라 구현은 복잡할 수 있지만, Databricks은 Databricks AI Search으로 이 과정을 단순화합니다. 이 섹션에서는 서비스의 아키텍처를 살펴보고 Delta Lake과의 원활한 통합 기능을 강조하여 자동 데이터 동기화를 진행합니다. 또한 Unity Catalog 아래 벡터 인덱스에 대한 안전하고 관리된 접근을 보장하는 통합 거버넌스 모델도 살펴볼 것입니다.

<!-- <img src="../Includes/images/03-vector-search-components.png" alt="Databricks AI Search components" /> -->
![03-vector-search-components](https://files.training.databricks.com/binder/prod_main/building-retrieval-agents-on-databricks-ko_kr-1.0.1/images/03-vector-search-components.png)

*그림 4. 이 도표는 Databricks AI Search*의 주요 구성 요소를 보여줍니다

### D1. 제품 개요

**Databricks AI Search**는 Databricks Lakehouse에 직접 통합된 벡터 데이터베이스 솔루션입니다. 이 확장 가능하고 지연 시간이 적은 서비스는 데이터의 벡터 표현과 메타데이터를 함께 저장하여 REST API 및 Python 클라이언트를 통한 실시간 유사도 검색을 가능하게 합니다. RAG 애플리케이션의 검색 성능을 최적화하기 위해 특별히 설계되어 별도의 벡터 데이터베이스 인프라를 관리할 필요가 없습니다.

### D2. Delta 동기화 및 인덱싱

Databricks AI Search의 가장 강력한 특징 중 하나는 **Delta Lake**와의 긴밀한 통합입니다. **Delta Sync API**를 통해 Vector Index가 자동으로 소스 Delta 테이블과 동기화됩니다. 소스 테이블에 데이터를 추가, 업데이트 또는 삭제할 때 Vector Index가 자동으로 업데이트되어, 검색 시스템이 항상 최신 데이터를 반영하도록 수동 개입 없이 보장합니다. 이로 인해 임베딩을 소스 데이터와 동기화하는 운영 부담이 줄어듭니다.

### D3. 관리 및 인제스트 모드

Databricks AI Search는 임베딩을 수집하고 관리하는 세 가지 유연한 접근법을 제공하여 필요에 맞는 제어 수준을 선택할 수 있습니다:

1. **관리되는 임베딩 (Delta 동기화):** 원시 텍스트가 포함된 소스 Delta 테이블을 제공하고, Databricks은 나머지를 처리합니다. 시스템은 구성된 **Mosaic AI Model Serving** endpoint(예: Foundation Model API)를 사용하여 임베딩을 자동으로 계산하고, 새로운 데이터를 처리하며, 인덱스를 업데이트합니다—임베딩 파이프라인을 관리할 필요가 없습니다.

1. **자가 관리 임베딩 (Delta 동기화):** 자신만의 커스텀 파이프라인을 사용해 임베딩을 compute 하여 Delta 테이블에 저장합니다. AI Search 인덱스는 이 표와 동기화되어 미리 계산된 벡터들을 색인화합니다. 이렇게 하면 임베딩 과정을 완전히 제어할 수 있으면서도 자동 동기화의 혜택을 누릴 수 있습니다.

1. **직접 접근 CRUD API:** AI Search 인덱스와 직접 상호작용할 수 있습니다 REST API 또는 Python SDK. 이를 통해 Delta 테이블 동기화에 의존하지 않고 벡터와 메타데이터를 직접 삽입, 업데이트 또는 삭제할 수 있어 실시간 애플리케이션이나 맞춤 Workflows에 이상적입니다.


<!-- <img src="../Includes/images/03-vector-search-managed-embeddings.png" alt="Databricks AI Search managed embeddings method" /> -->

![03-vector-search-managed-embeddings](https://files.training.databricks.com/binder/prod_main/building-retrieval-agents-on-databricks-ko_kr-1.0.1/images/03-vector-search-managed-embeddings.png)

*그림 5. 이 다이어그램은 AI Search이 자동 동기화를 통해 임베딩을 어떻게 관리하는지 보여줍니다.*

### D4. 거버넌스 및 접근 통제

Databricks AI Search는 **Unity Catalog**에 의해 관리되며, 데이터와 AI 자산 모두에 대해 **통합된 보안 모델을 제공합니다**. AI Search에서 생성된 인덱스는 Unity Catalog 내에서 보안 가능한 객체로 나타나, 관리자가 인덱스 수준에서 세분화된 접근 제어 목록(ACL)을 강제할 수 있게 합니다. 이로 인해 권한 있는 사용자와 애플리케이션만이 벡터 데이터를 쿼리하거나 수정할 수 있게 되어 전체 데이터 플랫폼에서 일관된 보안 정책을 유지할 수 있습니다.

## E. 요약

이번 강의에서는 RAG 시스템에서 데이터를 검색하기 위한 전체 라이프사이클을 탐구했습니다. 우리는 **임베딩**를 비구조화된 텍스트와 기계 판독 벡터 사이의 필수적인 다리로 정의했으며, 모델 선택이 특정 도메인과 쿼리 패턴에 맞춰져야 함을 강조했습니다. 우리는 **벡터 데이터베이스**의 메커니즘을 조사하고, 정확(KNN) 및 근사(ANN) 검색 전략을 구분했으며, **하이브리드 검색**과 **재순위조정**이 순수 벡터 유사성의 한계를 어떻게 극복하는지 발견하였습니다. 마지막으로, **Databricks AI Search**와 **Delta Sync**를 통한 임베딩 관리 자동화 기능과 **Unity Catalog**와의 강력한 보안 통합을 탐구했습니다.

**핵심 요점:**

1. **임베딩과 정렬:** 임베딩은 유사한 개념들을 벡터 공간에서 가까이 매핑하여 의미적 의미를 포착합니다. 효과적인 검색을 위해서는 임베딩 모델이 문서와 쿼리 모두에 대해 공유 벡터 공간을 만들어야 하며, 정렬을 위해 동일한 모델을 사용해야 합니다.
2. **검색 정밀도:** **ANN** 알고리즘이 생산 시스템에 필요한 속도와 확장성을 제공하지만, **reranker** 단계를 추가하는 것이 노이즈를 걸러내고 언어 모델에 높은 관련성을 보장하기 위해 종종 필수적입니다. 품질 향상과 추가된 지연 및 비용을 균형 있게 조정하세요.
3. **통합 아키텍처:** **Databricks AI Search**는 **Delta Lake**와의 자동 동기화와 유연한 인제스 모드(관리형, 자체 관리형 또는 직접 CRUD)를 지원하여 작업을 단순화합니다. 이 통합은 별도의 벡터 데이터베이스 인프라를 관리하는 복잡성을 없애면서도 Unity Catalog를 통한 엔터프라이즈급 거버넌스를 유지합니다.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>